In [ ]:
import sys
!{sys.executable} -m pip install -U "plotly>=6.1.1" "kaleido>=1.0.0"

  Using cached kaleido-1.2.0-py3-none-any.whl.metadata (5.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 68.6 MB/s eta 0:00:00
Using cached kaleido-1.2.0-py3-none-any.whl (68 kB)
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: kaleido
    Found existing installation: kaleido 0.2.1
    Uninstalling kaleido-0.2.1:
      Successfully uninstalled kaleido-0.2.1


In [31]:
import plotly, kaleido
import plotly.io as pio
print("plotly:", plotly.__version__)
print("kaleido:", kaleido.__version__)
print("kaleido scope:", pio.kaleido.scope is not None)

plotly: 5.24.1


AttributeError: module 'kaleido' has no attribute '__version__'

In [32]:
import sys, plotly
from importlib.metadata import version, PackageNotFoundError

print("python:", sys.version)
print("plotly:", plotly.__version__)

for pkg in ["kaleido", "plotly"]:
    try:
        print(pkg, "installed:", version(pkg))
    except PackageNotFoundError:
        print(pkg, "NOT installed")


python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
plotly: 5.24.1
kaleido installed: 0.2.1
plotly installed: 6.5.0


In [33]:
import sys
!{sys.executable} -m pip install -U "plotly>=6.1.1" "kaleido>=1.0.0"


  Using cached kaleido-1.2.0-py3-none-any.whl.metadata (5.6 kB)
Using cached kaleido-1.2.0-py3-none-any.whl (68 kB)
  Attempting uninstall: kaleido
    Found existing installation: kaleido 0.2.1
    Uninstalling kaleido-0.2.1:
      Successfully uninstalled kaleido-0.2.1


In [1]:
import plotly
import plotly.io as pio
from importlib.metadata import version

print("plotly:", plotly.__version__)
print("kaleido:", version("kaleido"))
print("kaleido scope is None?:", pio.kaleido.scope is None)


plotly: 6.5.0
kaleido: 1.2.0
kaleido scope is None?: False


In [4]:
import kaleido
kaleido.get_chrome_sync()

PosixPath('/usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

In [7]:
!apt update && apt-get install -y libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,411 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,966 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,598 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [69.2 kB

In [19]:
#-------------------------------------------------------------#
#----------Sunburst WITHOUT Gene Circle AND WITHOUT-----------#
#----------------------Sub-Categories-------------------------#
#----Grey Root + Synced Rotation + Multi-format Export--------#
#   SVG (transparent) + SVG (white) + PNG/TIFF 600dpi + PDF    #
#-------------------------------------------------------------#

import os
import copy
import pandas as pd
import plotly.express as px
import plotly.io as pio
import kaleido
kaleido.get_chrome_sync()
from plotly.subplots import make_subplots
from PIL import Image

# -----------------------------
# Settings
# -----------------------------
file_path = "Categorized_Gene_Table_HF_NF_with_LFC_no_unknown.csv"

ROOT_COLOR = "#BFBFBF"
ROTATION_DEG = 0
INSIDE_FONT_SIZE = 22

W, H = 1920, 1080
DPI = 600

OUT_HTML = "combined_sunburst_charts_No_GENE_No_SUBCAT.html"

OUT_SVG_TRANSPARENT = "combined_sunburst_charts_No_GENE_No_SUBCAT_transparent.svg"
OUT_SVG_WHITE       = "combined_sunburst_charts_No_GENE_No_SUBCAT_white.svg"
OUT_PNG_600         = "combined_sunburst_charts_No_GENE_No_SUBCAT_600dpi.png"
OUT_TIFF_600        = "combined_sunburst_charts_No_GENE_No_SUBCAT_600dpi.tiff"
OUT_PDF             = "combined_sunburst_charts_No_GENE_No_SUBCAT.pdf"

# -----------------------------
# Ensure Chrome for Kaleido v1+
# -----------------------------
def ensure_chrome_for_kaleido():
    """
    Kaleido v1+ needs Chrome/Chromium. If it's missing, install via kaleido helper.
    """
    try:
        import kaleido
        # Will raise if Chrome isn't available
        from kaleido.errors import ChromeNotFoundError
        try:
            # Quick probe: ask kaleido to launch a minimal session indirectly by checking a calc
            # Plotly will do it on first write_image; we just pre-install if needed.
            return True
        except ChromeNotFoundError:
            pass
    except Exception:
        # If kaleido isn't importable, plotly would fail anyway
        raise RuntimeError("Kaleido is not available. Install: pip install -U kaleido")

    # Install Chrome using kaleido's official helper
    try:
        import kaleido
        # Prefer sync in notebooks
        kaleido.get_chrome_sync()
        return True
    except Exception as e:
        raise RuntimeError(
            "Chrome is required for Kaleido image export but could not be installed automatically.\n"
            "Try running in a terminal:\n"
            "  plotly_get_chrome\n"
            "or in a notebook cell:\n"
            "  !plotly_get_chrome\n"
            f"\nDetails: {repr(e)}"
        )

# -----------------------------
# 1) Load + clean
# -----------------------------
df = pd.read_csv(file_path)

df["LFC"] = pd.to_numeric(df["LFC"], errors="coerce")
df = df.dropna(subset=["Morph", "Category", "LFC"]).copy()

df["Morph"] = df["Morph"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()
df["Gene Symbol"] = df["Gene Symbol"].astype(str).str.strip()

df["absLFC"] = df["LFC"].abs()

# -----------------------------
# 2) Aggregate to Category level only (Morph -> Category)
# -----------------------------
agg = (
    df.groupby(["Morph", "Category"], as_index=False)
      .agg(
          absLFC=("absLFC", "sum"),
          n_genes=("Gene Symbol", "nunique"),
      )
)

top5 = (
    df.sort_values(["Morph", "Category", "absLFC"], ascending=[True, True, False])
      .groupby(["Morph", "Category"])["Gene Symbol"]
      .apply(lambda s: ", ".join(s.head(5).astype(str)))
      .reset_index(name="top_genes")
)

agg = agg.merge(top5, on=["Morph", "Category"], how="left")
agg["Morph"] = agg["Morph"].astype(str).str.strip()
agg["Category"] = agg["Category"].astype(str).str.strip()
agg["top_genes"] = agg["top_genes"].fillna("")

# -----------------------------
# 3) Colors
# -----------------------------
all_categories = sorted(agg["Category"].unique())
palette = px.colors.qualitative.Plotly
category_color_map = {c: palette[i % len(palette)] for i, c in enumerate(all_categories)}

# -----------------------------
# 4) Build combined subplot figure (one per morph)
# -----------------------------
morphs = sorted(agg["Morph"].unique())
fig = make_subplots(
    rows=1,
    cols=len(morphs),
    specs=[[{"type": "domain"}] * len(morphs)],
    subplot_titles=[""] * len(morphs),
)

for col_i, morph in enumerate(morphs, start=1):
    agg_m = agg[agg["Morph"] == morph].copy()
    agg_m["Root"] = morph  # Option B: center circle

    fig_m = px.sunburst(
        agg_m,
        path=["Root", "Category"],
        values="absLFC",
        custom_data=["absLFC", "n_genes", "top_genes"],
    )

    trace = fig_m.data[0]
    ids = list(trace.ids)

    marker_colors = []
    for node_id in ids:
        parts = [p.strip() for p in str(node_id).split("/")]
        if len(parts) == 1:
            marker_colors.append(ROOT_COLOR)
        else:
            cat = parts[1]
            marker_colors.append(category_color_map.get(cat, ROOT_COLOR))

    fig_m.update_traces(
        marker=dict(colors=marker_colors),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Σ|LFC|: %{customdata[0]:.2f}<br>"
            "n genes: %{customdata[1]}<br>"
            "Top genes: %{customdata[2]}"
            "<extra></extra>"
        ),
        insidetextfont=dict(size=INSIDE_FONT_SIZE),
        insidetextorientation="radial",
        rotation=ROTATION_DEG,
        sort=False
    )

    fig.add_trace(fig_m.data[0], row=1, col=col_i)

# -----------------------------
# 5) Layout
# -----------------------------
fig.update_layout(
    title_text="Gene Expression in <i>S. pistillata</i> planulae morphs",
    title_x=0.5,
    title_font_size=36,
    width=W,
    height=H,
    font_family="serif",
    margin=dict(t=120, l=20, r=20, b=20),
)

fig.show()

# HTML export (interactive)
fig.write_html(OUT_HTML)
print(f"Saved HTML: {OUT_HTML}")

# -----------------------------
# 6) Static exports (SVG/PNG/TIFF/PDF)
# -----------------------------
# Kaleido must be active and Chrome must exist
if pio.kaleido.scope is None:
    raise RuntimeError("Kaleido is not active (pio.kaleido.scope is None).")

ensure_chrome_for_kaleido()

def with_bg(fig_in, transparent: bool):
    f = copy.deepcopy(fig_in)
    if transparent:
        f.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    else:
        f.update_layout(paper_bgcolor="white", plot_bgcolor="white")
    return f

# SVG: transparent + white
with_bg(fig, True).write_image(OUT_SVG_TRANSPARENT, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (transparent): {OUT_SVG_TRANSPARENT}")

with_bg(fig, False).write_image(OUT_SVG_WHITE, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (white): {OUT_SVG_WHITE}")

# PNG: embed 600 dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png)
im.save(OUT_PNG_600, dpi=(DPI, DPI))
im.close()
os.remove(tmp_png)
print(f"Saved PNG ({DPI} dpi metadata): {OUT_PNG_600}")

# TIFF: embed 600 dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png).convert("RGB")
im.save(OUT_TIFF_600, dpi=(DPI, DPI), compression="tiff_lzw")
im.close()
os.remove(tmp_png)
print(f"Saved TIFF ({DPI} dpi metadata): {OUT_TIFF_600}")

# PDF: vector
with_bg(fig, False).write_image(OUT_PDF, format="pdf", width=W, height=H, scale=1)
print(f"Saved PDF: {OUT_PDF}")


Saved HTML: combined_sunburst_charts_No_GENE_No_SUBCAT.html
Saved SVG (transparent): combined_sunburst_charts_No_GENE_No_SUBCAT_transparent.svg
Saved SVG (white): combined_sunburst_charts_No_GENE_No_SUBCAT_white.svg
Saved PNG (600 dpi metadata): combined_sunburst_charts_No_GENE_No_SUBCAT_600dpi.png
Saved TIFF (600 dpi metadata): combined_sunburst_charts_No_GENE_No_SUBCAT_600dpi.tiff
Saved PDF: combined_sunburst_charts_No_GENE_No_SUBCAT.pdf


In [14]:
#-------------------------------------------------------------#
#----------------Sunburst Without Gene Circle-----------------#
#----Grey Root + Synced Rotation + No Black Wedges + EXPORTS--#
#   SVG (transparent) + SVG (white) + PNG/TIFF 600dpi + PDF    #
#-------------------------------------------------------------#

import os
import copy
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from PIL import Image

# -----------------------------
# Settings
# -----------------------------
file_path = "Categorized_Gene_Table_HF_NF_with_LFC_no_unknown.csv"

ROOT_COLOR = "#BFBFBF"        # grey center
ROTATION_DEG = 0              # same rotation for HF and NF
INSIDE_FONT_SIZE = 22

W, H = 1920, 1080
DPI = 600

# One source of truth for ALL output names:
BASE_OUT = "combined_sunburst_charts_No_GENE"

HTML_OUT            = f"{BASE_OUT}.html"
SVG_TRANSPARENT_OUT = f"{BASE_OUT}_transparent.svg"
SVG_WHITE_OUT       = f"{BASE_OUT}_white.svg"
PNG_600_OUT         = f"{BASE_OUT}_600dpi.png"
TIFF_600_OUT        = f"{BASE_OUT}_600dpi.tiff"
PDF_OUT             = f"{BASE_OUT}.pdf"

# Kaleido must be active for static export
if pio.kaleido.scope is None:
    raise RuntimeError("Kaleido is not active (pio.kaleido.scope is None).")

# -----------------------------
# 1) Load + clean
# -----------------------------
print(f"[1/5] Loading CSV: {file_path}")
df = pd.read_csv(file_path)
print(f"     Loaded shape: {df.shape}")

df["LFC"] = pd.to_numeric(df["LFC"], errors="coerce")
before_drop = df.shape[0]
df = df.dropna(subset=["Morph", "Category", "Sub-Category", "LFC"]).copy()
after_drop = df.shape[0]
print(f"     After dropna: {after_drop} rows (dropped {before_drop - after_drop})")

df["Morph"] = df["Morph"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()
df["Sub-Category"] = df["Sub-Category"].astype(str).str.strip()
df["Gene Symbol"] = df["Gene Symbol"].astype(str).str.strip()

df["absLFC"] = df["LFC"].abs()
print(f"     Morphs: {sorted(df['Morph'].unique().tolist())}")
print(f"     Categories: {df['Category'].nunique()} | Sub-categories: {df['Sub-Category'].nunique()}")
print(f"     Σ absLFC (sanity): {df['absLFC'].sum():.3f}")

# -----------------------------
# 2) Aggregate (removes gene level)
# -----------------------------
print("[2/5] Aggregating to Morph -> Category -> Sub-Category")
agg = (
    df.groupby(["Morph", "Category", "Sub-Category"], as_index=False)
      .agg(
          absLFC=("absLFC", "sum"),
          n_genes=("Gene Symbol", "nunique"),
      )
)
print(f"     Aggregated rows: {agg.shape[0]}")

print("     Computing top genes (top 5 by absLFC) per sub-category")
top5 = (
    df.sort_values(["Morph", "Category", "Sub-Category", "absLFC"],
                   ascending=[True, True, True, False])
      .groupby(["Morph", "Category", "Sub-Category"])["Gene Symbol"]
      .apply(lambda s: ", ".join(s.head(5).astype(str)))
      .reset_index(name="top_genes")
)

agg = agg.merge(top5, on=["Morph", "Category", "Sub-Category"], how="left")
agg["Morph"] = agg["Morph"].astype(str).str.strip()
agg["Category"] = agg["Category"].astype(str).str.strip()
agg["Sub-Category"] = agg["Sub-Category"].astype(str).str.strip()
agg["top_genes"] = agg["top_genes"].fillna("")
print(f"     Σ absLFC over agg (sanity): {agg['absLFC'].sum():.3f}")

# -----------------------------
# 3) Colors
# -----------------------------
print("[3/5] Building category color map")
all_categories = sorted(agg["Category"].unique())
palette = px.colors.qualitative.Plotly
category_color_map = {c: palette[i % len(palette)] for i, c in enumerate(all_categories)}
print(f"     Mapped categories: {len(category_color_map)}")

# -----------------------------
# 4) Build combined subplot figure (one per morph)
# -----------------------------
morphs = sorted(agg["Morph"].unique())
print(f"[4/5] Building sunbursts for morphs: {morphs}")

fig = make_subplots(
    rows=1,
    cols=len(morphs),
    specs=[[{"type": "domain"}] * len(morphs)],
    subplot_titles=[""] * len(morphs),
)

for col_i, morph in enumerate(morphs, start=1):
    agg_m = agg[agg["Morph"] == morph].copy()
    agg_m["Root"] = morph  # Option B: center circle

    print(f"     -> Morph '{morph}': subcat rows={agg_m.shape[0]} | Σ absLFC={agg_m['absLFC'].sum():.3f}")

    fig_m = px.sunburst(
        agg_m,
        path=["Root", "Category", "Sub-Category"],
        values="absLFC",
        custom_data=["absLFC", "n_genes", "top_genes"],
    )

    trace = fig_m.data[0]
    ids = list(trace.ids)
    print(f"        Nodes rendered (root+categories+subcats): {len(ids)}")

    marker_colors = []
    for node_id in ids:
        parts = [p.strip() for p in str(node_id).split("/")]

        # Root
        if len(parts) == 1:
            marker_colors.append(ROOT_COLOR)
        # Category / Sub-Category -> inherit category color
        else:
            cat = parts[1]
            marker_colors.append(category_color_map.get(cat, ROOT_COLOR))

    fig_m.update_traces(
        marker=dict(colors=marker_colors),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Σ|LFC|: %{customdata[0]:.2f}<br>"
            "n genes: %{customdata[1]}<br>"
            "Top genes: %{customdata[2]}"
            "<extra></extra>"
        ),
        insidetextfont=dict(size=INSIDE_FONT_SIZE),
        insidetextorientation="radial",
        rotation=ROTATION_DEG,
        sort=False
    )

    fig.add_trace(fig_m.data[0], row=1, col=col_i)

# -----------------------------
# 5) Layout + export
# -----------------------------
print("[5/5] Rendering + exporting")
fig.update_layout(
    title_text="Gene Expression in <i>S. pistillata</i> planulae morphs",
    title_x=0.5,
    title_font_size=36,
    width=W,
    height=H,
    font_family="serif",
    margin=dict(t=120, l=20, r=20, b=20),
)

fig.show()

# HTML export (interactive)
fig.write_html(HTML_OUT)
print(f"Saved HTML: {HTML_OUT}")

def with_bg(fig_in, transparent: bool):
    f = copy.deepcopy(fig_in)
    if transparent:
        f.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    else:
        f.update_layout(paper_bgcolor="white", plot_bgcolor="white")
    return f

# SVG exports
with_bg(fig, True).write_image(SVG_TRANSPARENT_OUT, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (transparent): {SVG_TRANSPARENT_OUT}")

with_bg(fig, False).write_image(SVG_WHITE_OUT, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (white): {SVG_WHITE_OUT}")

# PNG export + 600dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png)
im.save(PNG_600_OUT, dpi=(DPI, DPI))
im.close()
os.remove(tmp_png)
print(f"Saved PNG ({DPI} dpi metadata): {PNG_600_OUT}")

# TIFF export + 600dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png).convert("RGB")
im.save(TIFF_600_OUT, dpi=(DPI, DPI), compression="tiff_lzw")
im.close()
os.remove(tmp_png)
print(f"Saved TIFF ({DPI} dpi metadata): {TIFF_600_OUT}")

# PDF export (vector; dpi not meaningful)
with_bg(fig, False).write_image(PDF_OUT, format="pdf", width=W, height=H, scale=1)
print(f"Saved PDF: {PDF_OUT}")


[1/5] Loading CSV: Categorized_Gene_Table_HF_NF_with_LFC_no_unknown.csv
     Loaded shape: (232, 8)
     After dropna: 232 rows (dropped 0)
     Morphs: ['HF', 'NF']
     Categories: 34 | Sub-categories: 60
     Σ absLFC (sanity): 459.750
[2/5] Aggregating to Morph -> Category -> Sub-Category
     Aggregated rows: 91
     Computing top genes (top 5 by absLFC) per sub-category
     Σ absLFC over agg (sanity): 459.750
[3/5] Building category color map
     Mapped categories: 34
[4/5] Building sunbursts for morphs: ['HF', 'NF']
     -> Morph 'HF': subcat rows=66 | Σ absLFC=413.260
        Nodes rendered (root+categories+subcats): 95
     -> Morph 'NF': subcat rows=25 | Σ absLFC=46.490
        Nodes rendered (root+categories+subcats): 44
[5/5] Rendering + exporting


Saved HTML: combined_sunburst_charts_No_GENE.html
Saved SVG (transparent): combined_sunburst_charts_No_GENE_transparent.svg
Saved SVG (white): combined_sunburst_charts_No_GENE_white.svg
Saved PNG (600 dpi metadata): combined_sunburst_charts_No_GENE_600dpi.png
Saved TIFF (600 dpi metadata): combined_sunburst_charts_No_GENE_600dpi.tiff
Saved PDF: combined_sunburst_charts_No_GENE.pdf


In [ ]:
#-------------------------------------------------------------#
#----------------Sunburst (WITH GENES)------------------------#
#----Synced Rotation HF/NF + No Black Wedges + EXPORTS--------#
#   HTML + SVG (transparent/white) + PNG/TIFF 600dpi + PDF     #
#-------------------------------------------------------------#

import os
import copy
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from PIL import Image

# -----------------------------
# Settings
# -----------------------------
file_path = "Categorized_Gene_Table_HF_NF_with_LFC_no_unknown.csv"

ROTATION_DEG = 0          # same start angle for HF & NF
INSIDE_FONT_SIZE = 22

W, H = 1920, 1080
DPI = 600

BASE_OUT = "combined_sunburst_charts_WITH_GENE"

HTML_OUT            = f"{BASE_OUT}.html"
SVG_TRANSPARENT_OUT = f"{BASE_OUT}_transparent.svg"
SVG_WHITE_OUT       = f"{BASE_OUT}_white.svg"
PNG_600_OUT         = f"{BASE_OUT}_600dpi.png"
TIFF_600_OUT        = f"{BASE_OUT}_600dpi.tiff"
PDF_OUT             = f"{BASE_OUT}.pdf"

# Kaleido must be active for static export
if pio.kaleido.scope is None:
    raise RuntimeError("Kaleido is not active (pio.kaleido.scope is None).")

# -----------------------------
# 1) Data Loading and Cleaning
# -----------------------------
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print("File not found. Using dummy data.")
    df = pd.DataFrame({
        "Morph": ["HF", "HF", "NF", "NF", "HF"],
        "Category": ["Adhesion", "Adhesion", "Immunity", "Immunity", "Signaling"],
        "Sub-Category": ["General", "Receptors", "Cytokines", "Receptors", "Kinases"],
        "Gene Symbol": ["GENE1", "GENE2", "GENE3", "GENE4", "GENE5"],
        "LFC": [2.5, -1.8, 3.1, -2.0, 1.5],
        "Description": ["Desc1", "Desc2", "Desc3", "Desc4", "Desc5"],
    })

df["LFC"] = pd.to_numeric(df["LFC"], errors="coerce")
df = df.dropna(subset=["Morph", "Category", "Sub-Category", "Gene Symbol", "LFC", "Description"]).copy()

# Normalize strings (strip whitespace)
df["Morph"] = df["Morph"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()
df["Sub-Category"] = df["Sub-Category"].astype(str).str.strip()
df["Gene Symbol"] = df["Gene Symbol"].astype(str).str.strip()
df["Description"] = df["Description"].astype(str).str.strip()

print("Data loaded and cleaned.")

# -----------------------------
# 2) Prepare Data for go.Sunburst
#    (Important fix for rotation comparability)
#    -> enforce consistent ordering of categories/subcats/genes
# -----------------------------
# Global ordering ensures HF and NF build wedges in the same sequence
category_order = sorted(df["Category"].unique())
subcat_order = (
    df[["Category", "Sub-Category"]].drop_duplicates()
      .sort_values(["Category", "Sub-Category"])
)
subcat_order_map = {c: [] for c in category_order}
for c in category_order:
    subcat_order_map[c] = subcat_order[subcat_order["Category"] == c]["Sub-Category"].tolist()

# Build the full sunburst arrays
sunburst_ids = []
sunburst_labels = []
sunburst_parents = []
sunburst_values = []
sunburst_hover_lfc_text = []
sunburst_hover_desc_text = []
sunburst_node_category = []

# Use a stable morph order (HF then NF if present)
morph_order = sorted(df["Morph"].unique().tolist())
if "HF" in morph_order and "NF" in morph_order:
    morph_order = ["HF", "NF"] + [m for m in morph_order if m not in ("HF", "NF")]

for morph in morph_order:
    morph_id = morph
    sunburst_ids.append(morph_id)
    sunburst_labels.append(morph)
    sunburst_parents.append("")
    sunburst_values.append(0)  # allowed; keeps center visible
    sunburst_hover_lfc_text.append("")
    sunburst_hover_desc_text.append("")
    sunburst_node_category.append(None)

    df_morph = df[df["Morph"] == morph]

    # enforce global category order
    for category in category_order:
        df_category = df_morph[df_morph["Category"] == category]
        if df_category.empty:
            continue

        category_id = f"{morph_id}_{category}"
        sunburst_ids.append(category_id)
        sunburst_labels.append(category)
        sunburst_parents.append(morph_id)
        sunburst_values.append(0)
        sunburst_hover_lfc_text.append("")
        sunburst_hover_desc_text.append("")
        sunburst_node_category.append(category)

        # enforce per-category subcat order
        for subcategory in subcat_order_map.get(category, []):
            df_subcategory = df_category[df_category["Sub-Category"] == subcategory]
            if df_subcategory.empty:
                continue

            subcategory_id = f"{category_id}_{subcategory}"
            sunburst_ids.append(subcategory_id)
            sunburst_labels.append(subcategory)
            sunburst_parents.append(category_id)
            sunburst_values.append(0)
            sunburst_hover_lfc_text.append("")
            sunburst_hover_desc_text.append("")
            sunburst_node_category.append(category)

            # enforce stable gene order within subcategory (by abs(LFC) desc, then name)
            df_subcategory = df_subcategory.assign(absLFC=df_subcategory["LFC"].abs())
            df_subcategory = df_subcategory.sort_values(["absLFC", "Gene Symbol"], ascending=[False, True])

            for _, row in df_subcategory.iterrows():
                gene_symbol = row["Gene Symbol"]
                gene_id = f"{subcategory_id}_{gene_symbol}"
                sunburst_ids.append(gene_id)
                sunburst_labels.append(gene_symbol)
                sunburst_parents.append(subcategory_id)
                sunburst_values.append(float(abs(row["LFC"])))
                sunburst_hover_lfc_text.append(f"LFC: {row['LFC']:.2f}")
                sunburst_hover_desc_text.append(f"Description: {row['Description']}")
                sunburst_node_category.append(category)

sunburst_data_df = pd.DataFrame({
    "ids": sunburst_ids,
    "labels": sunburst_labels,
    "parents": sunburst_parents,
    "values": sunburst_values,
    "hover_lfc_text": sunburst_hover_lfc_text,
    "hover_desc_text": sunburst_hover_desc_text,
    "node_category": sunburst_node_category
})
print("Sunburst data DataFrame created.")

# -----------------------------
# 3) Generate Individual go.Sunburst Charts (one per morph)
# -----------------------------
morph_go_charts = {}

unique_morphs_from_data = sunburst_data_df["ids"][sunburst_data_df["parents"] == ""].unique()

all_unique_categories = category_order
palette = px.colors.qualitative.Plotly
category_color_map = {c: palette[i % len(palette)] for i, c in enumerate(all_unique_categories)}

for morph_name in unique_morphs_from_data:
    df_filtered_for_morph = sunburst_data_df[
        (sunburst_data_df["ids"] == morph_name) |
        (sunburst_data_df["ids"].str.startswith(f"{morph_name}_"))
    ].copy()

    node_colors = []
    for category_val in df_filtered_for_morph["node_category"]:
        if category_val is None:
            node_colors.append("#cccccc")
        else:
            node_colors.append(category_color_map.get(str(category_val).strip(), "#cccccc"))

    customdata_list = df_filtered_for_morph[["hover_lfc_text", "hover_desc_text"]].values.tolist()

    hovertemplate = (
        "<b>%{label}</b><br>"
        "%{customdata[0]}<br>"
        "%{customdata[1]}<extra></extra>"
    )

    sunburst_trace = go.Sunburst(
        ids=df_filtered_for_morph["ids"],
        labels=df_filtered_for_morph["labels"],
        parents=df_filtered_for_morph["parents"],
        values=df_filtered_for_morph["values"],
        customdata=customdata_list,
        hovertemplate=hovertemplate,
        marker=dict(colors=node_colors),
        insidetextfont=dict(size=INSIDE_FONT_SIZE),
        rotation=ROTATION_DEG,   # <-- synced rotation
        sort=False               # <-- critical: prevent Plotly re-sorting wedges
    )

    fig_m = go.Figure(sunburst_trace)
    fig_m.update_layout(title_text=f"{morph_name} Gene Expression Sunburst Chart")
    morph_go_charts[morph_name] = fig_m

print(f"Generated {len(morph_go_charts)} individual go.Sunburst charts.")

# -----------------------------
# 4) Combine and Display Sunburst Charts
# -----------------------------
num_charts = len(morph_go_charts)
fig = make_subplots(
    rows=1,
    cols=num_charts,
    specs=[[{"type": "domain"}] * num_charts],
    subplot_titles=[""] * num_charts
)

# Keep subplot order stable using morph_order
ordered_keys = [m for m in morph_order if m in morph_go_charts]
for i, morph_name in enumerate(ordered_keys):
    fig.add_trace(morph_go_charts[morph_name].data[0], row=1, col=i + 1)

fig.update_layout(
    title_text="Gene Expression in <i>S. pistillata</i> planulae morphs",
    title_x=0.5,
    title_font_size=36,
    width=W,
    height=H,
    font_family="serif",
    margin=dict(t=120, l=20, r=20, b=20)
)

fig.show()
print("Combined Sunburst charts displayed.")

# -----------------------------
# 5) Exports (same set as before)
# -----------------------------
fig.write_html(HTML_OUT)
print(f"Saved HTML: {HTML_OUT}")

def with_bg(fig_in, transparent: bool):
    f = copy.deepcopy(fig_in)
    if transparent:
        f.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    else:
        f.update_layout(paper_bgcolor="white", plot_bgcolor="white")
    return f

# SVG: transparent + white
with_bg(fig, True).write_image(SVG_TRANSPARENT_OUT, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (transparent): {SVG_TRANSPARENT_OUT}")

with_bg(fig, False).write_image(SVG_WHITE_OUT, format="svg", width=W, height=H, scale=1)
print(f"Saved SVG (white): {SVG_WHITE_OUT}")

# PNG: export then embed 600 dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png)
im.save(PNG_600_OUT, dpi=(DPI, DPI))
im.close()
os.remove(tmp_png)
print(f"Saved PNG ({DPI} dpi metadata): {PNG_600_OUT}")

# TIFF: export then convert to TIFF with 600 dpi metadata
tmp_png = "_tmp_export.png"
with_bg(fig, False).write_image(tmp_png, format="png", width=W, height=H, scale=1)
im = Image.open(tmp_png).convert("RGB")
im.save(TIFF_600_OUT, dpi=(DPI, DPI), compression="tiff_lzw")
im.close()
os.remove(tmp_png)
print(f"Saved TIFF ({DPI} dpi metadata): {TIFF_600_OUT}")

# PDF: vector (dpi not meaningful)
with_bg(fig, False).write_image(PDF_OUT, format="pdf", width=W, height=H, scale=1)
print(f"Saved PDF: {PDF_OUT}")


Data loaded and cleaned.
Sunburst data DataFrame created.
Generated 2 individual go.Sunburst charts.


Combined Sunburst charts displayed.
Saved HTML: combined_sunburst_charts_WITH_GENE.html
Saved SVG (transparent): combined_sunburst_charts_WITH_GENE_transparent.svg
Saved SVG (white): combined_sunburst_charts_WITH_GENE_white.svg
Saved PNG (600 dpi metadata): combined_sunburst_charts_WITH_GENE_600dpi.png
Saved TIFF (600 dpi metadata): combined_sunburst_charts_WITH_GENE_600dpi.tiff
Saved PDF: combined_sunburst_charts_WITH_GENE.pdf
